In [1]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
raw_text = (
    "Our first task is to familiarize ourselves with the framework we gonna use through most of the laboratories."
)
input_ids = tokenizer(raw_text, return_tensors="pt").input_ids.to(DEVICE)

print(f"Token count: {input_ids.shape[1]}")
print(f"Decoded prompt: {tokenizer.decode(input_ids[0])}")

Token count: 20
Decoded prompt: Our first task is to familiarize ourselves with the framework we gonna use through most of the laboratories.


In [4]:
generation_kwargs = {
    "min_new_tokens": 50,
    "max_new_tokens": 50,
    "do_sample": False,
}

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.time()

with torch.no_grad():
    output_ids = model.generate(input_ids, **generation_kwargs)

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.time()
duration = end_time - start_time

generated_ids = output_ids[0][input_ids.shape[1] :]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("===== Generated Text =====")
print(generated_text)
print("==========================")
print(f"Generation time: {duration:.2f}s")
print(f"Throughput: {len(generated_ids) / duration:.2f} tokens/s")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


===== Generated Text =====
 This is the Python programming language. We will use Python 3.7 or higher. We will use the following libraries: numpy, pandas, matplotlib, and seaborn. We will also use the following tools: Jupyter Notebook, and the Jupyter
Generation time: 1.28s
Throughput: 39.10 tokens/s
